# Phase 2: Baseline & Prompt Engineering
## Nemotron Reasoning Challenge

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

## 2.1 Check Available Resources

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2.2 Load Test Data

In [ ]:
test_df = pd.read_csv('data/test.csv')
print(f"Test samples: {len(test_df)}")
for i, row in test_df.iterrows():
    print(f"\n--- Test {i+1} (ID: {row['id']}) ---")
    print(row['prompt'][:300])

## 2.3 Design Prompt Templates

In [ ]:
# Zero-shot prompt template
ZERO_SHOT_TEMPLATE = """Solve the following puzzle. Show your reasoning step by step.

{prompt}

Provide only the final answer."""

# Few-shot with chain-of-thought
FEW_SHOT_COT_TEMPLATE = """Solve the following puzzle by explaining your reasoning step by step.

{prompt}

Let's think step by step:"""

# Enhanced prompt with extraction instructions
EXTRACT_TEMPLATE = """Analyze this puzzle carefully. 

{prompt}

Provide the final answer in this format: Answer: <your answer>"""

## 2.4 Test with Pretrained Model (Optional - depends on GPU)

In [ ]:
# This cell is optional - depends on having GPU memory for the model
# If running on limited GPU, we may need to use a smaller model or skip this

# model_name = "nvidia/Nemotron-3-Nano-8B"  # Smaller version if 30B doesn't fit
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

## 2.5 Alternative: Rule Extraction & Solving
Since the puzzles follow specific transformation rules, we can attempt to solve them programmatically.

In [ ]:
# For bit manipulation puzzles, try to infer the rule from examples

def extract_bit_examples(prompt):
    """Extract input->output examples from prompt."""
    pattern = r'([01]{8})\s*->\s*([01]{8})'
    return re.findall(pattern, prompt)

def solve_bit_puzzle(prompt, target_input):
    """Try to find the bit manipulation rule and apply to target."""
    examples = extract_bit_examples(prompt)
    
    if not examples:
        return None
    
    # Build mapping from input patterns
    # Try common transformations and see which one matches the examples
    
    operations = [
        ("reverse", lambda x: int(bin(x)[2:].zfill(8)[::-1], 2)),
        ("complement", lambda x: (~x) & 0xFF),
        ("swap_nibbles", lambda x: ((x & 0xF0) >> 4) | ((x & 0x0F) << 4)),
        ("rotate_left_1", lambda x: ((x << 1) | (x >> 7)) & 0xFF),
        ("rotate_right_1", lambda x: ((x >> 1) | (x << 7)) & 0xFF),
    ]
    
    for name, op in operations:
        matches = 0
        for inp, out in examples:
            inp_int = int(inp, 2)
            out_int = int(out, 2)
            if op(inp_int) == out_int:
                matches += 1
        
        if matches == len(examples) and len(examples) > 0:
            # Found a matching rule
            target_int = int(target_input, 2)
            result = op(target_int)
            return format(result, '08b')
    
    return None

## 2.6 Prepare Data for Fine-tuning

In [ ]:
# Create formatted training data for fine-tuning
train_df = pd.read_csv('data/train.csv')

def format_for_training(row):
    """Format prompt + answer for training."""
    prompt = row['prompt']
    answer = row['answer']
    return {
        'prompt': prompt,
        'answer': answer,
        'text': f"Instruction: Solve the puzzle.\n\n{prompt}\n\nAnswer: {answer}"
    }

# Sample some examples
formatted_data = train_df.head(100).apply(format_for_training, axis=1).tolist()
print("Sample formatted example:")
print(formatted_data[0]['text'][:500])

## Summary
- Designed several prompt templates (zero-shot, few-shot CoT, extract)
- Created rule extraction helpers for bit puzzles
- Prepared data formatting for fine-tuning
- Next step: Set up NeMo for LoRA fine-tuning